In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import CRUD Python module
from CRUD_Python_Module import AnimalShelter


###########################
# Data Manipulation / Model
###########################
# Hardcoded MongoDB authentication for dashboard access

username = "aacuser"
password = "aacpass"
db = AnimalShelter(username, password)

# Convert MongoDB query results into a Pandas DataFrame for use in the dashboard
df = pd.DataFrame.from_records(db.read({}))

# Remove MongoDB ObjectId field since it is not compatible with Dash DataTable
if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# Load Grazioso Salvare logo and encode for display
image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read()).decode()

app.layout = html.Div([
    # Hidden div placeholder (kept for potential state handling)
    html.Div(id='hidden-div', style={'display': 'none'}),
    
    # Header section with logo, title, and identifier
    html.Center([
        html.A(
            html.Img(
                src='data:image/png;base64,{}'.format(encoded_image),
                style={'height': '150px'}
            ),
            href='https://www.snhu.edu',
            target='_blank'
        ),
        html.H1('Grazioso Salvare Dashboard'),
        html.H3('Tyler Shurilla')
    ]),
    
    html.Hr(),
    
    # Filter selection for rescue types
    html.Div([
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Reset', 'value': 'reset'},
                {'label': 'Water Rescue', 'value': 'water'},
                {'label': 'Mountain or Wilderness Rescue', 'value': 'mountain'},
                {'label': 'Disaster or Individual Tracking', 'value': 'disaster'}
            ],
            value='reset',
            inline=True
    )
    ], style={'marginBottom': '20px'}),
    
    html.Hr(),
    
    # Data table displaying animal records
    dash_table.DataTable(
        id='datatable-id',
        columns=[
            {"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns
        ],
        data=df.to_dict('records'),
        
        # Carryover from Module 6 (kept as-is since it was already working)
        row_selectable='single',
        selected_rows=[0],
        page_size=10,
        sort_action='native',
        filter_action='native',
        
        # Basic usability improvements
        style_table={'overflowX': 'auto'},
        style_cell={
            'textAlign': 'left',
            'padding': '5px',
            'minWidth': '120px',
            'width': '120px',
            'maxWidth': '180px',
            'whiteSpace': 'normal'
        },
        style_header={
            'backgroundColor': 'rgb(230, 230, 230)',
            'fontWeight': 'bold'
        }
    ),
    
    html.Br(),
    html.Hr(),
    
    # Layout for chart and map side-by-side
    html.Div(
        className='row',
        style={'display': 'flex'},
        children=[
            html.Div(
                id='graph-id',
                className='col s12 m6',
            ),
            html.Div(
                id='map-id',
                className='col s12 m6',
            )
        ]
    )
])



#############################################
# Interaction Between Components / Controller
#############################################

@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):
    # Default view returns the full dataset
    if filter_type == 'reset':
        query = {}
    
    # Water rescue candidates
    elif filter_type == 'water':
        query = {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "Labrador Retriever Mix",
                    "Chesapeake Bay Retriever",
                    "Newfoundland"
                ]
            },
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {
                "$gte": 26,
                "$lte": 156
            }
        }
    
    # Mountain or wilderness rescue candidates
    elif filter_type == 'mountain':
        query = {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "German Shepherd",
                    "Alaskan Malamute",
                    "Old English Sheepdog",
                    "Siberian Husky",
                    "Rottweiler"
                ]
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {
                "$gte": 26,
                "$lte": 156
            }
        }
    
    # Disaster rescue or individual tracking candidates
    elif filter_type == 'disaster':
        query = {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "Doberman Pinscher",
                    "German Shepherd",
                    "Golden Retriever",
                    "Bloodhound",
                    "Rottweiler"
                ]
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {
                "$gte": 20,
                "$lte": 300
            }
        }
    
    # Fallback to full dataset if something unexpected gets passed in
    else:
        query = {}

    dff = pd.DataFrame.from_records(db.read(query))

    # Remove ObjectId field before sending records to the table
    if '_id' in dff.columns:
        dff.drop(columns=['_id'], inplace=True)

    return dff.to_dict('records')


# Build the secondary chart from the currently displayed table data
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")]
)
def update_graphs(viewData):
    if viewData is None or len(viewData) == 0:
        return []

    dff = pd.DataFrame.from_dict(viewData)

    # Count breeds from the current table view
    breed_counts = dff['breed'].value_counts().reset_index()
    breed_counts.columns = ['breed', 'count']

    # Keep the chart readable by grouping smaller categories into Other
    if len(breed_counts) > 10:
        top_breeds = breed_counts.head(10).copy()
        other_count = breed_counts['count'].iloc[10:].sum()

        other_row = pd.DataFrame([{
            'breed': 'Other',
            'count': other_count
        }])

        chart_data = pd.concat([top_breeds, other_row], ignore_index=True)
    else:
        chart_data = breed_counts

    return [
        dcc.Graph(
            figure=px.pie(
                chart_data,
                names='breed',
                values='count',
                title='Breed Distribution'
            )
        )
    ]


# Highlight the selected row so the map interaction is easier to follow
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_rows')]
)
def update_styles(selected_rows):
    if selected_rows is None or len(selected_rows) == 0:
        return []

    return [{
        'if': {'row_index': selected_rows[0]},
        'backgroundColor': '#D2F3FF'
    }]


# Update the geolocation map from the selected row in the current table view
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")]
)
def update_map(viewData, index):
    if viewData is None or len(viewData) == 0:
        return []

    dff = pd.DataFrame.from_dict(viewData)

    # Default to the first row if nothing is selected
    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]

    return [
        dl.Map(
            style={'width': '1000px', 'height': '500px'},
            center=[30.75, -97.48],
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),
                dl.Marker(
                    position=[dff.iloc[row, 13], dff.iloc[row, 14]],
                    children=[
                        dl.Tooltip(dff.iloc[row, 4]),
                        dl.Popup([
                            html.H1("Animal Name"),
                            html.P(dff.iloc[row, 9])
                        ])
                    ]
                )
            ]
        )
    ]


# Run the dashboard in JupyterLab
app.run_server()

Dash app running on https://lunchsheriff-inventtotal-3000.codio.io/proxy/8050/
